# Retrieval descriptor diagnostic — scaffold split, full test set

Scaffold-split companion to `fig_retrieval_descriptor_diagnostic.ipynb`'s
"SI — full test set (n=27,647), no RI restriction" section. Same plot
convention: recall and top-1 accuracy vs. median per-query candidate pool
size, MW and heavy-atom (HA) filters overlaid, dashed grey line = unfiltered
(no-filter) baseline on the same query set.

No RI-restricted panels here (unlike the random-split notebook's StdNP-subset
panels): the AIRI RI eval TSV used elsewhere is built from the random split's
test set, so RI-based filtering/ranking would conflate two different splits
-- see `fig_retrieval_results_pubchem_global_scaffold.ipynb`'s intro for the
same caveat. MW here uses the full fine-grained sweep (±0 through ±10 Da,
plus ±80 Da) run for scaffold; HA uses the 5 windows run so far (±1, 2, 3,
6, 8).

MassFormer is out of scope for the scaffold PubChem run.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from icicle.utils.visualization.style import (
    get_cmap,
    get_palette,
    make_fig,
    save_fig,
    set_style,
)

set_style("manuscript")
palette = get_palette()

RESULTS_DIR = Path("../../results/pubchem_retrieval_eval_icicle_scaffold_s1")
OUTPUT_DIR = Path("../../figures/retrieval_descriptor_diagnostic_scaffold")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LIBRARY_SIZE = 93_661_074

## Load per-query data — full test set (n=31,263), no RI restriction

Auto-discovers whichever MW/HA windows have result files on disk, rather
than hardcoding a fixed window list, so this notebook re-runs correctly as
more windows are added later without edits.

In [ ]:
def discover_windows(prefix):
    pattern = re.compile(rf"^retrieval_per_query_{prefix}(\d+)_all\.tsv$")
    tags = []
    for p in RESULTS_DIR.glob(f"retrieval_per_query_{prefix}*_all.tsv"):
        m = pattern.match(p.name)
        if m:
            tags.append(m.group(1))
    return sorted(tags, key=int)


mw_tags = discover_windows("mw")
ha_tags = discover_windows("heavy_atom")
print(f"MW windows found: {mw_tags}")
print(f"HA windows found: {ha_tags}")

mw_dfs_all = {
    f"\u00b1{tag} Da": pd.read_csv(
        RESULTS_DIR / f"retrieval_per_query_mw{tag}_all.tsv", sep="\t"
    )
    for tag in mw_tags
}
ha_dfs_all = {
    f"\u00b1{tag}": pd.read_csv(
        RESULTS_DIR / f"retrieval_per_query_heavy_atom{tag}_all.tsv", sep="\t"
    )
    for tag in ha_tags
}

with open(RESULTS_DIR / "retrieval_global_results.json") as fh:
    global_results = json.load(fh)
unfiltered_top1_all = global_results["all"]["autofail"]["cosine"][
    "top_1_accuracy"
]
n_queries = pd.read_csv(
    RESULTS_DIR / "retrieval_global_per_query.tsv", sep="\t"
).shape[0]
print(f"n_queries (full test set) = {n_queries}")

## Panel A — recall vs. median candidate pool size

Recall = fraction of queries whose true molecule falls inside the candidate
window (upper bound on achievable top-1 under autofail scoring).

In [ ]:
def pool_and_recall(df):
    return df["n_candidates"].median(), df["true_mol_found"].mean()


def sorted_xy(points):
    xs = [p[0] for p in points]
    ys = [100 * p[1] for p in points]
    order = sorted(range(len(xs)), key=lambda i: xs[i])
    return [xs[i] for i in order], [ys[i] for i in order]


mw_recall_points = [pool_and_recall(df) for df in mw_dfs_all.values()]
ha_recall_points = [pool_and_recall(df) for df in ha_dfs_all.values()]

fig, ax = make_fig("square")
for points, label, color, marker in [
    (mw_recall_points, f"MW (n={n_queries:,})", palette[3], "s"),
    (ha_recall_points, f"Heavy-atom count (n={n_queries:,})", palette[5], "^"),
]:
    xs, ys = sorted_xy(points)
    ax.plot(xs, ys, marker=marker, color=color, label=label, linewidth=1.5)

ax.set_xscale("log")
ax.set_xlabel("Median candidate pool size")
ax.set_ylabel("Recall (%)")
ax.set_ylim(0, 100)
ax.legend(fontsize=7)

save_fig(fig, "recall_vs_pool_size_full_testset_scaffold", OUTPUT_DIR)
plt.show()
plt.close(fig)

## Panel B — top-1 accuracy vs. median candidate pool size

Dashed horizontal line = unfiltered top-1 on the same full test set
(`retrieval_global_results.json["all"]["autofail"]["cosine"]`) -- plotted as
a reference line, not a point on the curve, since its true candidate-set
size (~93.6M, the whole database) is off the natural range of the MW/HA
window sweep.

In [ ]:
def pool_and_top1(df):
    return df["n_candidates"].median(), (
        df["rank_autofail_cosine"] <= 1
    ).mean()


mw_top1_points = [pool_and_top1(df) for df in mw_dfs_all.values()]
ha_top1_points = [pool_and_top1(df) for df in ha_dfs_all.values()]

fig, ax = make_fig("square")
for points, label, color, marker in [
    (mw_top1_points, "Peak-mass window", palette[3], "s"),
    (ha_top1_points, "Heavy-atom count", palette[5], "^"),
]:
    xs, ys = sorted_xy(points)
    ax.plot(xs, ys, marker=marker, color=color, label=label, linewidth=1.5)

ax.axhline(
    100 * unfiltered_top1_all,
    color="gray",
    linestyle="--",
    linewidth=1.0,
    label="Unfiltered",
)

ax.set_xscale("log")
ax.set_xlabel("Median candidate pool size")
ax.set_ylabel("Top-1 accuracy (%)")
ax.legend(fontsize=7)

save_fig(fig, "top1_vs_pool_size_full_testset_scaffold", OUTPUT_DIR)
plt.show()
plt.close(fig)

### Same plot, coverage encoded as marker color

Same axes and data as above, but each marker is colored by its window's
coverage (recall) -- the fraction of queries whose true molecule actually
survives that filter -- using one shared colorbar across both series.
Series identity comes from marker shape only (square = peak-mass window,
triangle = heavy-atom count); a thin line at fixed low opacity connects
each series' own points purely to show ordering, not to carry any
encoding itself. Color reads far more clearly than opacity here: the
heavy-atom count line is one continuous streak from dark purple (~29%
coverage at ±1) to yellow (~98% at ±8), directly showing why its
early points underperform -- the true molecule is simply missing from the
pool for most queries there, not being outranked within it.

In [ ]:
def pool_and_recall(df):
    return df["n_candidates"].median(), df["true_mol_found"].mean()


mw_recall_points = [pool_and_recall(df) for df in mw_dfs_all.values()]
ha_recall_points = [pool_and_recall(df) for df in ha_dfs_all.values()]


def sorted_xyc(points_xy, points_recall):
    xs = [p[0] for p in points_xy]
    ys = [100 * p[1] for p in points_xy]
    cov = [
        100 * p[1] for p in points_recall
    ]  # percent, same order as points_xy
    order = sorted(range(len(xs)), key=lambda i: xs[i])
    return (
        [xs[i] for i in order],
        [ys[i] for i in order],
        [cov[i] for i in order],
    )


fig, ax = make_fig("square")
cmap = get_cmap().reversed()

mappable = None
for top1_points, recall_points, label, marker in [
    (mw_top1_points, mw_recall_points, "Peak-mass window", "s"),
    (ha_top1_points, ha_recall_points, "Heavy-atom count", "^"),
]:
    xs, ys, cov = sorted_xyc(top1_points, recall_points)
    ax.plot(xs, ys, color="grey", alpha=0.35, linewidth=1.5, zorder=1)
    mappable = ax.scatter(
        xs,
        ys,
        c=cov,
        cmap=cmap,
        vmin=0,
        vmax=100,
        marker=marker,
        s=55,
        zorder=2,
    )
    # legend proxy: marker shape only, neutral color (color itself is on the colorbar, not the legend)
    ax.plot(
        [],
        [],
        color="grey",
        marker=marker,
        linestyle="none",
        markersize=6,
        label=label,
    )

ax.axhline(
    100 * unfiltered_top1_all,
    color="gray",
    linestyle="--",
    linewidth=1.0,
    label="Unfiltered",
)

ax.set_xscale("log")
ax.set_xlabel("Median candidate pool size")
ax.set_ylabel("Top-1 accuracy (%)")
ax.legend(fontsize=7)
cbar = fig.colorbar(mappable, ax=ax)
cbar.set_label("Coverage (%)", fontsize=8)
cbar.ax.tick_params(labelsize=7)

save_fig(
    fig, "top1_vs_pool_size_full_testset_scaffold_coverage_color", OUTPUT_DIR
)
plt.show()
plt.close(fig)

## Numeric summary

In [ ]:
def rows_for(dfs, filter_name):
    out = []
    for label, df in dfs.items():
        pool, recall = pool_and_recall(df)
        _, top1 = pool_and_top1(df)
        out.append(
            {
                "filter": filter_name,
                "window": label,
                "n": len(df),
                "median_cand": pool,
                "recall_pct": 100 * recall,
                "top1_pct": 100 * top1,
            }
        )
    return out


rows = rows_for(mw_dfs_all, "MW") + rows_for(ha_dfs_all, "HA")
rows.append(
    {
        "filter": "Unfiltered",
        "window": "all",
        "n": n_queries,
        "median_cand": LIBRARY_SIZE,
        "recall_pct": 100.0,
        "top1_pct": 100 * unfiltered_top1_all,
    }
)

df_summary = pd.DataFrame(rows)
df_summary.to_csv(
    OUTPUT_DIR / "descriptor_diagnostic_summary_scaffold.csv", index=False
)
df_summary

In [ ]:
def pool_and_topk(df, k):
    return df["n_candidates"].median(), (
        df["rank_autofail_cosine"] <= k
    ).mean()


for K in (10, 50):
    topk_points_mw = [pool_and_topk(df, K) for df in mw_dfs_all.values()]
    topk_points_ha = [pool_and_topk(df, K) for df in ha_dfs_all.values()]
    unfiltered_topk_all = global_results["all"]["autofail"]["cosine"][
        f"top_{K}_accuracy"
    ]

    fig, ax = make_fig("square")
    cmap = get_cmap().reversed()
    mappable = None
    for topk_pts, recall_points, label, marker in [
        (topk_points_mw, mw_recall_points, "Peak-mass window", "s"),
        (topk_points_ha, ha_recall_points, "Heavy-atom count", "^"),
    ]:
        xs, ys, cov = sorted_xyc(topk_pts, recall_points)
        ax.plot(xs, ys, color="grey", alpha=0.35, linewidth=1.5, zorder=1)
        mappable = ax.scatter(
            xs,
            ys,
            c=cov,
            cmap=cmap,
            vmin=0,
            vmax=100,
            marker=marker,
            s=55,
            zorder=2,
        )
        ax.plot(
            [],
            [],
            color="grey",
            marker=marker,
            linestyle="none",
            markersize=6,
            label=label,
        )

    ax.axhline(
        100 * unfiltered_topk_all,
        color="gray",
        linestyle="--",
        linewidth=1.0,
        label="Unfiltered",
    )

    ax.set_xscale("log")
    ax.set_xlabel("Median candidate pool size")
    ax.set_ylabel(f"Top-{K} accuracy (%)")
    ax.legend(fontsize=7)
    cbar = fig.colorbar(mappable, ax=ax)
    cbar.set_label("Coverage (%)", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    save_fig(
        fig,
        f"top{K}_vs_pool_size_full_testset_scaffold_coverage_color",
        OUTPUT_DIR,
    )
    plt.show()
    plt.close(fig)